# Demo 3: Group chat — multiple agents talking to each other

**Demo 1** used two agents in a **1:1** chat (`UserProxy` ↔ `AssistantAgent`).  
**Demo 2** showed a **single** custom assistant with its own memory and reply logic.

Here we use **`GroupChat`** + **`GroupChatManager`**: several assistants (plus a user proxy) share one thread. The manager uses the LLM to decide **who speaks next** (`speaker_selection_method="auto"`), so roles can debate, refine ideas, and hand off naturally.

In [2]:
import os
import autogen
from dotenv import load_dotenv
load_dotenv()



True

## LLM config (same idea as Demo 1)

All agents that call the model share this `llm_config`. Set `OPENAI_API_KEY` in `.env`.

In [3]:
MODEL = os.getenv("AUTOGEN_MODEL", "gpt-4o-mini")

llm_config = {
    "config_list": [
        {
            "model": MODEL,
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    ],
    "temperature": 0.7,
}


## Agents with different roles

- **User_Proxy** — starts the task (`human_input_mode="NEVER"` so the notebook runs without typing at the console; switch to `"ALWAYS"` if you want to join live).
- **Product_Manager** — scope, users, success criteria.
- **Engineering_Lead** — architecture, APIs, implementation tradeoffs.
- **Security_Reviewer** — threats, data handling, least privilege.

They only "communicate" by posting in the same group chat; the **GroupChatManager** picks the next speaker.

In [5]:
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    human_input_mode="ALWAYS",
    max_consecutive_auto_reply=8,
    code_execution_config={"use_docker": False},
    system_message=(
        "You represent the project owner. You asked for a short plan; listen to the team, "
        "ask at most one clarifying follow-up if needed, then summarize agreements. "
        'When the group has a good enough answer, end with the word TERMINATE on its own line.'
    ),
    
)


product = autogen.AssistantAgent(
    name="Product_Manager",
    llm_config=llm_config,
    system_message=(
        "You are a senior engineer. Propose a minimal architecture (components, data flow). "
        "Flag complexity and dependencies; do not write huge code blocks—sketches only."
    ),
    code_execution_config={"use_docker": False},

)


engineering = autogen.AssistantAgent(
    name="Engineering_Lead",
    llm_config=llm_config,
    system_message=(
        "You are a senior engineer. Propose a minimal architecture (components, data flow). "
        "Flag complexity and dependencies; do not write huge code blocks—sketches only."
    ),
    code_execution_config={"use_docker": False},

)

security = autogen.AssistantAgent(
    name="Security_Reviewer",
    llm_config=llm_config,
    system_message=(
        "You are a security-minded reviewer. Call out top risks (authn/z, secrets, PII, supply chain). "
        "Suggest concrete mitigations; stay brief and actionable."
    ),
    code_execution_config={"use_docker": False},

)

## Group chat + manager

`GroupChatManager` uses the same LLM to route the conversation (`speaker_selection_method="auto"`).  
Tune **`max_round`** to cap cost; increase if you want longer debates.

In [6]:
agents = [user_proxy, product, engineering, security]

groupchat = autogen.GroupChat(
    agents=agents,
    messages=[],
    max_round=18,
    speaker_selection_method="auto",
    allow_repeat_speaker=True,
)


manager = autogen.GroupChatManager(
    name="GroupChat_Manager",
    groupchat=groupchat,
    llm_config=llm_config,
    code_execution_config={"use_docker": False},
)

## Run the discussion

The user proxy **initiates** chat with the **manager** (not a single assistant). The manager broadcasts messages inside the group.

In [ ]:
task = (
    "We want a small internal web app where employees upload CSV expense rows and get a monthly PDF summary. "
    "No mobile app. Discuss scope, a simple architecture, and the main security checks. "
    "End with a bullet summary the owner can approve."
)

user_proxy.initiate_chat(manager, message=task)



User_Proxy (to GroupChat_Manager):

We want a small internal web app where employees upload CSV expense rows and get a monthly PDF summary. No mobile app. Discuss scope, a simple architecture, and the main security checks. End with a bullet summary the owner can approve.

--------------------------------------------------------------------------------

Next speaker: Product_Manager

Product_Manager (to GroupChat_Manager):

### Scope
The internal web application will allow employees to:
- Upload CSV files containing expense data.
- Process and validate the uploaded data.
- Generate a monthly summary in PDF format.
- Provide a simple user interface for uploading files and viewing summaries.

### Architecture Components
1. **Frontend (Web Interface)**
   - A simple HTML/CSS/JavaScript UI allowing file uploads and summary downloads.
   - A progress indicator for uploads and processing.

2. **Backend (Server)**
   - **API Layer**: RESTful API to handle file uploads and retrieval of summarie